In [1]:
# ============================================
# Rumah Adat Classification - ResNet50 (torchvision)
# ============================================

import os, json, random
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from torchvision import datasets, transforms, models
from torchvision.transforms import functional as TF

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import f1_score, classification_report, confusion_matrix

In [2]:
# -----------------------------
# 0. Reproducibility & Device
# -----------------------------
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = 2
BATCH_SIZE = 32
NUM_EPOCHS = 20

# -----------------------------
# 1. Path & Dataset (ImageFolder)
# -----------------------------
train_dir = "/kaggle/input/datanya/Train/Train"
test_dir  = "/kaggle/input/datanya/Test/Test"


# load tanpa transform dulu untuk ambil label
base_no_tf = datasets.ImageFolder(train_dir, transform=None)
class_names = base_no_tf.classes
num_classes = len(class_names)
print("Kelas:", class_names)

with open("label_mapping.json","w") as f:
    json.dump({i:c for i,c in enumerate(class_names)}, f)

all_labels = [y for _, y in base_no_tf.samples]

# -----------------------------
# 2. Transforms (224) + Augmentasi
#    Pakai weights ResNet50_Weights.IMAGENET1K_V2
# -----------------------------
weights = models.ResNet50_Weights.IMAGENET1K_V2

# coba ambil mean/std dari weights; fallback ke ImageNet default
try:
    mean, std = weights.meta["mean"], weights.meta["std"]
except Exception:
    mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

# Val/test transform (aman & konsisten dgn pralatih)
val_transform = weights.transforms()  # Resize(232)->CenterCrop(224), ToTensor, Normalize

# Train transform (augmentasi ringan + normalisasi ImageNet)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomResizedCrop(224, scale=(0.9, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

# -----------------------------
# 3. Stratified Split 80/20
# -----------------------------
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, val_idx = next(sss.split(np.zeros(len(all_labels)), all_labels))

class TransformingSubset(torch.utils.data.Dataset):
    def __init__(self, base_dataset, indices, transform):
        self.base = base_dataset
        self.indices = indices
        self.transform = transform
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        idx = self.indices[i]
        path, target = self.base.samples[idx]
        img = self.base.loader(path)  # PIL
        img = self.transform(img) if self.transform else img
        return img, target

base_for_tf = datasets.ImageFolder(train_dir, transform=None)
train_ds = TransformingSubset(base_for_tf, train_idx, train_transform)
val_ds   = TransformingSubset(base_for_tf, val_idx,   val_transform)

# -----------------------------
# 4. Imbalance Handling (opsional)
# -----------------------------
train_labels = [all_labels[i] for i in train_idx]
class_counts = np.bincount(train_labels, minlength=num_classes)
class_weights = 1.0 / (class_counts + 1e-6)
sample_weights = [class_weights[y] for y in train_labels]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

# -----------------------------
# 5. DataLoader
# -----------------------------
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

# -----------------------------
# 6. Model ResNet50 + Head Kustom
# -----------------------------
model = models.resnet50(weights=weights)
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, num_classes)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

# -----------------------------
# 7. Training Loop (macro-F1 selection)
# -----------------------------
best_val_f1 = 0.0
best_model_path = "best_resnet50.pth"

for epoch in range(1, NUM_EPOCHS+1):
    # Train
    model.train()
    tr_loss = 0.0
    tr_preds, tr_targets = [], []

    for x, y in tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} [Train]"):
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits = model(x)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()

        tr_loss += loss.item() * x.size(0)
        tr_preds.extend(logits.argmax(1).detach().cpu().numpy())
        tr_targets.extend(y.detach().cpu().numpy())

    tr_loss /= len(train_ds)
    tr_acc = (np.array(tr_preds) == np.array(tr_targets)).mean()
    tr_f1 = f1_score(tr_targets, tr_preds, average="macro")

    # Val
    model.eval()
    va_loss = 0.0
    va_preds, va_targets = [], []
    with torch.no_grad():
        for x, y in tqdm(val_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} [Val]"):
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            logits = model(x)
            loss = criterion(logits, y)
            va_loss += loss.item() * x.size(0)
            va_preds.extend(logits.argmax(1).detach().cpu().numpy())
            va_targets.extend(y.detach().cpu().numpy())

    va_loss /= len(val_ds)
    va_acc = (np.array(va_preds) == np.array(va_targets)).mean()
    va_f1 = f1_score(va_targets, va_preds, average="macro")

    scheduler.step()

    print(f"\nEpoch {epoch}/{NUM_EPOCHS}")
    print(f"  Train - Loss: {tr_loss:.4f} | Acc: {tr_acc:.4f} | F1(macro): {tr_f1:.4f}")
    print(f"  Val   - Loss: {va_loss:.4f} | Acc: {va_acc:.4f} | F1(macro): {va_f1:.4f}")

    if va_f1 > best_val_f1:
        best_val_f1 = va_f1
        torch.save(model.state_dict(), best_model_path)
        print("  ✅ Model terbaik disimpan (berdasarkan Val F1)")

# -----------------------------
# 8. Laporan per kelas
# -----------------------------
print("\n=== Laporan Klasifikasi (Validation) ===")
print(classification_report(va_targets, va_preds, target_names=class_names, digits=4))
cm = confusion_matrix(va_targets, va_preds)
print("Confusion Matrix (baris=gt, kolom=pred):\n", cm)

# -----------------------------
# 9. Inference + TTA (flip horizontal)
# -----------------------------
# Muat best checkpoint
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
model.eval()

def predict_with_tta_resnet(img_pil, model, device, val_tf):
    variants = []
    # original
    x0 = val_tf(img_pil).unsqueeze(0).to(device)
    variants.append(x0)
    # hflip
    x1 = val_tf(TF.hflip(img_pil)).unsqueeze(0).to(device)
    variants.append(x1)

    with torch.no_grad():
        probs = []
        for x in variants:
            p = torch.softmax(model(x), dim=1)
            probs.append(p)
        p_mean = torch.stack(probs, dim=0).mean(dim=0)  # [1, C]
        pred = p_mean.argmax(1).item()
    return pred

test_images = sorted(os.listdir(test_dir))
ids, preds = [], []

for img_name in tqdm(test_images, desc="Predicting Test (TTA)"):
    img_path = os.path.join(test_dir, img_name)
    img_pil = datasets.folder.default_loader(img_path)
    pred_idx = predict_with_tta_resnet(img_pil, model, DEVICE, val_transform)
    ids.append(os.path.splitext(img_name)[0])
    preds.append(class_names[pred_idx])

submission = pd.DataFrame({"id": ids, "style": preds}).sort_values("id")
submission.to_csv("submission.csv", index=False)
print("✅ submission.csv tersimpan.")
print(f"Best Val F1 (macro): {best_val_f1:.4f}")
print(submission.head())

Kelas: ['balinese', 'batak', 'dayak', 'javanese', 'minangkabau']


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 221MB/s]
/tmp/ipykernel_19/2930643341.py:110: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
Epoch 1/20 [Train]:   0%|          | 0/44 [00:00<?, ?it/s]/tmp/ipykernel_19/2930643341.py:128: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
Epoch 1/20 [Val]: 100%|██████████| 11/11 [00:48<00:00,  4.42s/it]



Epoch 1/20
  Train - Loss: 0.9847 | Acc: 0.7066 | F1(macro): 0.7084
  Val   - Loss: 0.9547 | Acc: 0.7265 | F1(macro): 0.6697
  ✅ Model terbaik disimpan (berdasarkan Val F1)


Epoch 2/20 [Train]:   0%|          | 0/44 [00:00<?, ?it/s]/tmp/ipykernel_19/2930643341.py:128: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
Epoch 2/20 [Val]: 100%|██████████| 11/11 [00:43<00:00,  3.92s/it]



Epoch 2/20
  Train - Loss: 0.5453 | Acc: 0.8822 | F1(macro): 0.8830
  Val   - Loss: 0.7955 | Acc: 0.7892 | F1(macro): 0.7065
  ✅ Model terbaik disimpan (berdasarkan Val F1)


...


Epoch 19/20
  Train - Loss: 0.2641 | Acc: 0.9829 | F1(macro): 0.9824
  Val   - Loss: 0.7727 | Acc: 0.8262 | F1(macro): 0.7358


Epoch 20/20 [Train]:   0%|          | 0/44 [00:00<?, ?it/s]/tmp/ipykernel_19/2930643341.py:128: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
Epoch 20/20 [Val]: 100%|██████████| 11/11 [00:43<00:00,  4.00s/it]



Epoch 20/20
  Train - Loss: 0.2562 | Acc: 0.9879 | F1(macro): 0.9875
  Val   - Loss: 0.7600 | Acc: 0.8433 | F1(macro): 0.7624
  ✅ Model terbaik disimpan (berdasarkan Val F1)

=== Laporan Klasifikasi (Validation) ===
              precision    recall  f1-score   support

    balinese     0.8834    0.9290    0.9057       155
       batak     0.9091    0.5263    0.6667        19
       dayak     0.6923    0.6429    0.6667        14
    javanese     0.7727    0.6800    0.7234        50
 minangkabau     0.8250    0.8761    0.8498       113

    accuracy                         0.8433       351
   macro avg     0.8165    0.7309    0.7624       351
weighted avg     0.8426    0.8433    0.8392       351

Confusion Matrix (baris=gt, kolom=pred):
 [[144   0   0   5   6]
 [  1  10   1   2   5]
 [  1   0   9   1   3]
 [  8   0   1  34   7]
 [  9   1   2   2  99]]


Predicting Test (TTA): 100%|██████████| 444/444 [02:11<00:00,  3.36it/s]


✅ submission.csv tersimpan.
Best Val F1 (macro): 0.7624
         id        style
0  Test_001     balinese
1  Test_002  minangkabau
2  Test_003     javanese
3  Test_004     balinese
4  Test_005     balinese
